In [1]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl

from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import compute_epi_mask, apply_mask, unmask
from nilearn.image import index_img, resample_to_img, math_img, mean_img, get_data
from nilearn.plotting import plot_stat_map, show
from nilearn import image, masking
from nilearn import plotting
from nilearn.datasets import load_mni152_brain_mask
from pandas import read_excel

from nilearn.glm.second_level import SecondLevelModel
from scipy.stats import norm

from nilearn.glm import threshold_stats_img

from pathlib import Path

In [6]:
WhichStory = 'shapesphysical'

textValueFileDF = pd.read_excel(f"/Users/silvycollin/Documents/GitHub/wholeBrain_narratives_MAC_MFT/text/foundationScores/{WhichStory}_MFT_MAC.xlsx")

virtues = textValueFileDF[['MAC_a_fairness_virtue',
                               'MAC_a_group_virtue',
                               'MAC_a_deference_virtue',
                               'MAC_a_heroism_virtue',
                               'MAC_a_reciprocity_virtue',
                               'MAC_a_family_virtue',
                               'MAC_a_property_virtue']]

vices = textValueFileDF[['MAC_a_fairness_vice',
                               'MAC_a_group_vice',
                               'MAC_a_deference_vice',
                               'MAC_a_heroism_vice',
                               'MAC_a_reciprocity_vice',
                               'MAC_a_family_vice',
                               'MAC_a_property_vice']]

# output directory
base_dir1 = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/"
)

output_dir = Path(f"{base_dir1}/CountPhrases/")
output_dir


PosixPath('/Users/silvycollin/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/CountPhrases')

In [11]:
textValueFileDF['sentence'][1]

'On the morning of June 15th, Guy Burkhardt woke up screaming.'

In [15]:
textValueFileDF['sentence'][2]

'It was more real than any dream he had ever had in his life.'

In [3]:

events = pd.DataFrame( {"trial_type":range(0,len(virtues))})

# dtermine the max column for each row
max_column = virtues.idxmax(axis=1)

# Determine rows where all 7 columns are 0.00 (to two decimal places)
all_zero_mask = (virtues.round(2) == 0).all(axis=1)
# Replace column name with "baseline" for those rows
max_column[all_zero_mask] = "baseline"

max_columns_list = max_column.tolist()

events['trial_type'] = max_columns_list

In [4]:
events

,trial_type
0,baseline
1,MAC_a_reciprocity_virtue
2,MAC_a_family_virtue
3,MAC_a_group_virtue
4,baseline
...,...
1137,baseline
1138,MAC_a_fairness_virtue
1139,MAC_a_property_virtue
1140,MAC_a_family_virtue


In [5]:
events_vice = pd.DataFrame( {"trial_type":range(0,len(vices))})

# dtermine the max column for each row
max_column = vices.idxmax(axis=1)

# Determine rows where all 7 columns are 0.00 (to two decimal places)
all_zero_mask = (vices.round(2) == 0).all(axis=1)
# Replace column name with "baseline" for those rows
max_column[all_zero_mask] = "baseline"

max_columns_list = max_column.tolist()

events_vice['trial_type'] = max_columns_list

In [6]:
events_vice

,trial_type
0,baseline
1,MAC_a_fairness_vice
2,MAC_a_fairness_vice
3,MAC_a_reciprocity_vice
4,baseline
...,...
1137,baseline
1138,MAC_a_family_vice
1139,MAC_a_fairness_vice
1140,MAC_a_deference_vice


In [7]:
baseline_count = (events['trial_type'] == 'baseline').sum()
fai_count = (events['trial_type'] == 'MAC_a_fairness_virtue').sum()
fam_count = (events['trial_type'] == 'MAC_a_family_virtue').sum()
her_count = (events['trial_type'] == 'MAC_a_heroism_virtue').sum()
rec_count = (events['trial_type'] == 'MAC_a_reciprocity_virtue').sum()
gr_count = (events['trial_type'] == 'MAC_a_group_virtue').sum()
prop_count = (events['trial_type'] == 'MAC_a_property_virtue').sum()
def_count = (events['trial_type'] == 'MAC_a_deference_virtue').sum()

baseline_count_V = (events_vice['trial_type'] == 'baseline').sum()
fai_count_V = (events_vice['trial_type'] == 'MAC_a_fairness_vice').sum()
fam_count_V = (events_vice['trial_type'] == 'MAC_a_family_vice').sum()
her_count_V = (events_vice['trial_type'] == 'MAC_a_heroism_vice').sum()
rec_count_V = (events_vice['trial_type'] == 'MAC_a_reciprocity_vice').sum()
gr_count_V = (events_vice['trial_type'] == 'MAC_a_group_vice').sum()
prop_count_V = (events_vice['trial_type'] == 'MAC_a_property_vice').sum()
def_count_V = (events_vice['trial_type'] == 'MAC_a_deference_vice').sum()


# Create a dataframe
countedPhrasesPerFoundation = pd.DataFrame({
    'foundation': [
        'baseline',
        'fairness',
        'family',
        'heroism',
        'reciprocity',
        'group',
        'property',
        'deference'
    ],
    'virtue': [
        baseline_count,
        fai_count,
        fam_count,
        her_count,
        rec_count,
        gr_count,
        prop_count,
        def_count
    ],
    'vice': [
        baseline_count_V,
        fai_count_V,
        fam_count_V,
        her_count_V,
        rec_count_V,
        gr_count_V,
        prop_count_V,
        def_count_V
    ]
})

# Save dataframe to CSV
countedPhrasesPerFoundation.to_csv(str(output_dir)+f'/{WhichStory}_CountPhrases.csv', index=False)



In [8]:
countedPhrasesPerFoundation

,foundation,virtue,vice
0,baseline,366,367
1,fairness,112,243
2,family,164,67
3,heroism,72,121
4,reciprocity,119,95
5,group,190,97
6,property,31,65
7,deference,88,87
